# Bulk Expression Viewer

Interactive gene expression viewer for the processed organoid RNA-seq matrices in this repository.

Use it in Google Colab after opening the repo or copying this notebook into your private Colab workspace.

Capabilities:
- query one gene or a comma-separated gene list
- show all samples or a custom list of sample numbers
- switch between `counts`, `TPM`, `log2(TPM+1)`, and `VST`
- aggregate by `line`, `condition`, `project_group`, or `passage`
- render sample-level barplots, passage trends, and heatmaps


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

repo_root = Path('.').resolve()
sys.path.insert(0, str((repo_root / 'src').resolve()))

from bulk_expression_viewer import (
    export_render_bundle,
    get_scale_matrix,
    load_project_data,
    matrix_to_long,
    parse_gene_input,
    parse_sample_input,
    plot_aggregated_heatmap,
    plot_gene_barplot,
    plot_multi_gene_heatmap,
    plot_passage_trend,
    resolve_genes,
    setup_plot_style,
)

setup_plot_style()
project = load_project_data(repo_root / 'data' / 'processed')

print(f"Loaded counts/TPM/VST matrices with {project.tpm.shape[0]:,} genes and {project.tpm.shape[1]} samples.")
display(project.metadata[['sample_id', 'sample_name', 'line', 'condition', 'project_group', 'passage']].head(10))


In [ ]:
scale_widget = widgets.Dropdown(
    options=['counts', 'TPM', 'log2(TPM+1)', 'VST'],
    value='TPM',
    description='Scale:',
    style={'description_width': 'initial'},
)

genes_widget = widgets.Textarea(
    value='EPCAM, VIM, MKI67',
    description='Genes:',
    placeholder='Enter one gene or a comma-separated list',
    layout=widgets.Layout(width='100%', height='90px'),
    style={'description_width': 'initial'},
)

sample_mode_widget = widgets.Dropdown(
    options=['All samples', 'Custom sample numbers'],
    value='All samples',
    description='Samples:',
    style={'description_width': 'initial'},
)

samples_widget = widgets.Textarea(
    value='29, 30, 31, 32, 33, 34, 35, 36, 37, 38',
    description='Sample numbers:',
    placeholder='Example: 1, 2, 3, 29, 30',
    layout=widgets.Layout(width='100%', height='70px'),
    style={'description_width': 'initial'},
    disabled=True,
)

aggregate_widget = widgets.Checkbox(
    value=False,
    description='Add aggregated heatmap',
    indent=False,
)

groupby_widget = widgets.Dropdown(
    options=[('Line', 'line'), ('Condition', 'condition'), ('Project group', 'project_group'), ('Passage', 'passage')],
    value='line',
    description='Aggregate by:',
    style={'description_width': 'initial'},
)

aggfunc_widget = widgets.Dropdown(
    options=['mean', 'median'],
    value='mean',
    description='Reducer:',
    style={'description_width': 'initial'},
)

table_rows_widget = widgets.IntSlider(
    value=20,
    min=5,
    max=100,
    step=5,
    description='Table rows:',
    style={'description_width': 'initial'},
)

render_button = widgets.Button(
    description='Render plots',
    button_style='warning',
    icon='bar-chart',
)
export_button = widgets.Button(
    description='Export current selection',
    button_style='success',
    icon='download',
)
output = widgets.Output()
last_render = {}


def _toggle_sample_box(change):
    samples_widget.disabled = change['new'] != 'Custom sample numbers'


sample_mode_widget.observe(_toggle_sample_box, names='value')


def render_dashboard(_=None):
    global last_render
    with output:
        clear_output(wait=True)
        try:
            requested_genes = parse_gene_input(genes_widget.value)
            if not requested_genes:
                raise ValueError('Please provide at least one gene symbol.')

            if sample_mode_widget.value == 'All samples':
                sample_keys = project.metadata['sample_key'].tolist()
            else:
                sample_keys = parse_sample_input(samples_widget.value, project.metadata['sample_id'].tolist())

            matrix = get_scale_matrix(scale_widget.value, project)
            resolved_genes, suggestions = resolve_genes(requested_genes, matrix.index)

            if not resolved_genes:
                suggestion_lines = []
                for gene, matches in suggestions.items():
                    if matches:
                        suggestion_lines.append(f'- `{gene}`: try {", ".join(matches)}')
                suggestion_text = '\n'.join(suggestion_lines) if suggestion_lines else 'No close matches found.'
                raise ValueError(f'No gene symbols were resolved.\n\n{suggestion_text}')

            selected_meta = project.metadata[project.metadata['sample_key'].isin(sample_keys)].copy()
            selected_meta = selected_meta.sort_values('sample_id')
            sample_keys = selected_meta['sample_key'].tolist()

            long_df = matrix_to_long(matrix, selected_meta, resolved_genes, sample_keys)
            matrix_subset = matrix.loc[resolved_genes, sample_keys]
            figures = {}

            summary = (
                f"**Resolved genes:** {', '.join(resolved_genes)}  \n"
                f"**Samples shown:** {len(sample_keys)}  \n"
                f"**Scale:** `{scale_widget.value}`"
            )
            display(Markdown(summary))

            unresolved = [gene for gene in requested_genes if gene not in resolved_genes]
            if unresolved:
                unresolved_lines = []
                for gene in unresolved:
                    matches = suggestions.get(gene, [])
                    unresolved_lines.append(f'- `{gene}`: {", ".join(matches) if matches else "no close match"}')
                display(Markdown('**Unresolved genes**\n' + '\n'.join(unresolved_lines)))

            if len(resolved_genes) == 1:
                fig = plot_gene_barplot(long_df, scale_widget.value)
                figures['sample_barplot'] = fig
                display(fig)
                plt.close(fig)

                if long_df['passage'].notna().sum() >= 2:
                    fig = plot_passage_trend(long_df, scale_widget.value)
                    figures['passage_trend'] = fig
                    display(fig)
                    plt.close(fig)

            fig = plot_multi_gene_heatmap(matrix_subset, selected_meta, scale_widget.value)
            figures['heatmap'] = fig
            display(fig)
            plt.close(fig)

            if aggregate_widget.value:
                fig = plot_aggregated_heatmap(long_df, groupby_widget.value, aggfunc_widget.value, scale_widget.value)
                figures['aggregated_heatmap'] = fig
                display(fig)
                plt.close(fig)

            preview = long_df[
                ['gene_symbol', 'sample_id', 'sample_name', 'line', 'condition', 'project_group', 'passage', 'expression']
            ].sort_values(['gene_symbol', 'sample_id'])
            display(preview.head(table_rows_widget.value).reset_index(drop=True))

            last_render = {
                'long_df': preview.copy(),
                'matrix_subset': matrix_subset.copy(),
                'metadata_subset': selected_meta.copy(),
                'figures': figures,
            }

            export_dir = repo_root / 'data' / 'exports'
            display(Markdown(f'Export target: `{export_dir}`'))

        except Exception as exc:
            display(Markdown(f'**Error:** {exc}'))


def export_current_selection(_=None):
    with output:
        if not last_render:
            display(Markdown('**Error:** render plots first, then export.'))
            return

        bundle_dir = export_render_bundle(
            long_df=last_render['long_df'],
            matrix_subset=last_render['matrix_subset'],
            metadata_subset=last_render['metadata_subset'],
            figures=last_render['figures'],
            output_dir=repo_root / 'data' / 'exports',
        )
        display(Markdown(f'**Exported to:** `{bundle_dir}`'))


controls = widgets.VBox([
    scale_widget,
    genes_widget,
    sample_mode_widget,
    samples_widget,
    widgets.HBox([aggregate_widget, groupby_widget, aggfunc_widget]),
    table_rows_widget,
    widgets.HBox([render_button, export_button]),
])

render_button.on_click(render_dashboard)
export_button.on_click(export_current_selection)
display(controls, output)
render_dashboard()
